# RNN + Embeddings from ESM-2

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from contextlib import nullcontext

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 1. Load Dataset

In [ ]:
df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv')
if 'len' not in df.columns: df['len'] = df['seq'].str.len()
print(df.head()); df.info()

## 2. Load ESM-2 and Generate Embeddings

In [ ]:
esm_model, alphabet = torch.hub.load('facebookresearch/esm:main', 'esm2_t30_150M_UR50D')
esm_model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

sequences = [s.replace('*','X') for s in df['seq'].tolist()]
labels = df['pdb_id'].tolist() if 'pdb_id' in df.columns else list(range(len(sequences)))
pairs = list(zip(labels, sequences))

all_embeddings = []
for i in tqdm(range(0, len(pairs), 8), desc='Generating Embeddings'):
    batch = pairs[i:i+8]
    _, _, toks = batch_converter(batch)
    toks = toks.to(device)
    with torch.no_grad():
        out = esm_model(toks, repr_layers=[esm_model.num_layers], return_contacts=False)
    emb = out['representations'][esm_model.num_layers][:,1:-1,:]
    all_embeddings.extend([e.cpu() for e in emb])

padded_embeddings = pad_sequence(all_embeddings, batch_first=True, padding_value=0.0)
embedding_dim = padded_embeddings.shape[-1]
padded_embeddings.shape, embedding_dim

## 3. Encode Labels and Dataloaders

In [ ]:
def encode_labels(ss_labels, vocab, max_len):
    enc = []
    for ss in ss_labels:
        ids = [vocab.get(c, -1) for c in ss]
        if len(ids) < max_len: ids.extend([-1]*(max_len-len(ids)))
        else: ids = ids[:max_len]
        enc.append(torch.tensor(ids, dtype=torch.long))
    return pad_sequence(enc, batch_first=True, padding_value=-1)

seq_pad_len = padded_embeddings.shape[1]
ss8_labels = encode_labels(df['sst8'], ss8_vocab, seq_pad_len)
ss3_labels = encode_labels(df['sst3'], ss3_vocab, seq_pad_len)

class ProteinDataset(Dataset):
    def __init__(self, emb, s8, s3): self.emb, self.s8, self.s3 = emb, s8, s3
    def __len__(self): return len(self.emb)
    def __getitem__(self, i): return self.emb[i], self.s8[i], self.s3[i]

train_idx, tmp_idx = train_test_split(range(len(padded_embeddings)), test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.5, random_state=42)
train_ds = ProteinDataset(padded_embeddings[train_idx], ss8_labels[train_idx], ss3_labels[train_idx])
val_ds   = ProteinDataset(padded_embeddings[val_idx],   ss8_labels[val_idx],   ss3_labels[val_idx])
test_ds  = ProteinDataset(padded_embeddings[test_idx],  ss8_labels[test_idx],  ss3_labels[test_idx])
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)
embedding_dim

## 4. RNN Model

In [ ]:
class ProteinRNN(nn.Module):
    def __init__(self, input_dim=640, hidden_dim=256, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=layers, bidirectional=False, batch_first=True, dropout=dropout if layers>1 else 0.0)
        self.out_dropout = nn.Dropout(dropout)
        self.q8_head = nn.Linear(hidden_dim, 8)
        self.q3_head = nn.Linear(hidden_dim, 3)
    def forward(self, x, lengths=None):
        if lengths is None:
            lengths = torch.full((x.size(0),), x.size(1), dtype=torch.int64)
        orig_len = x.size(1)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out,_ = self.lstm(packed)
        x,_ = pad_packed_sequence(packed_out, batch_first=True, total_length=orig_len)
        x = self.out_dropout(x)
        return self.q8_head(x), self.q3_head(x)

model = ProteinRNN(input_dim=embedding_dim).to(device)
if torch.cuda.device_count()>1: model = nn.DataParallel(model)
model

## 5. Training

In [ ]:
def compute_accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    mask = labels >= 0
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 20
best_val_acc_q8 = 0.0
for epoch in range(num_epochs):
    model.train(); train_loss=train_acc_q8=train_acc_q3=0.0
    for emb, ss8, ss3 in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        emb, ss8, ss3 = emb.to(device), ss8.to(device), ss3.to(device)
        lengths = (ss8 >= 0).sum(dim=1).to(torch.int64)
        optimizer.zero_grad()
        q8_logits, q3_logits = model(emb, lengths=lengths)
        l8 = criterion_q8(q8_logits.view(-1,8), ss8.view(-1))
        l3 = criterion_q3(q3_logits.view(-1,3), ss3.view(-1))
        loss = l8 + 0.5*l3
        loss.backward(); optimizer.step()
        train_loss += loss.item(); train_acc_q8 += compute_accuracy(q8_logits, ss8); train_acc_q3 += compute_accuracy(q3_logits, ss3)
    train_loss/=len(train_loader); train_acc_q8/=len(train_loader); train_acc_q3/=len(train_loader)
    model.eval(); val_loss=val_acc_q8=val_acc_q3=0.0
    with torch.no_grad():
        for emb, ss8, ss3 in val_loader:
            emb, ss8, ss3 = emb.to(device), ss8.to(device), ss3.to(device)
            lengths = (ss8 >= 0).sum(dim=1).to(torch.int64)
            q8_logits, q3_logits = model(emb, lengths=lengths)
            l8 = criterion_q8(q8_logits.view(-1,8), ss8.view(-1))
            l3 = criterion_q3(q3_logits.view(-1,3), ss3.view(-1))
            loss = l8 + 0.5*l3
            val_loss += loss.item(); val_acc_q8 += compute_accuracy(q8_logits, ss8); val_acc_q3 += compute_accuracy(q3_logits, ss3)
    val_loss/=len(val_loader); val_acc_q8/=len(val_loader); val_acc_q3/=len(val_loader)
    print(f'Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}')
    print(f'Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}')
    print(f'Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}')
    if val_acc_q8>best_val_acc_q8:
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(state, 'best_rnn_esm2.pt'); best_val_acc_q8 = val_acc_q8

# Test
eval_model = ProteinRNN(input_dim=embedding_dim).to(device)
eval_model.load_state_dict(torch.load('best_rnn_esm2.pt', map_location='cpu'))
if torch.cuda.device_count()>1: eval_model = nn.DataParallel(eval_model)
eval_model.eval(); test_loss=test_acc_q8=test_acc_q3=0.0
with torch.no_grad():
    for emb, ss8, ss3 in test_loader:
        emb, ss8, ss3 = emb.to(device), ss8.to(device), ss3.to(device)
        q8_logits, q3_logits = eval_model(emb)
        l8 = criterion_q8(q8_logits.view(-1,8), ss8.view(-1))
        l3 = criterion_q3(q3_logits.view(-1,3), ss3.view(-1))
        loss = l8 + 0.5*l3
        test_loss += loss.item(); test_acc_q8 += compute_accuracy(q8_logits, ss8); test_acc_q3 += compute_accuracy(q3_logits, ss3)
test_loss/=len(test_loader); test_acc_q8/=len(test_loader); test_acc_q3/=len(test_loader)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy Q8: {test_acc_q8:.4f}')
print(f'Test Accuracy Q3: {test_acc_q3:.4f}')
